# Tema 6 - Osciladores

**Electronica General - 2o GIERM**

---

## Objetivos de aprendizaje

- Comprender el concepto de oscilador como amplificador inestable con realimentacion positiva
- Aplicar el criterio de Barkhausen para determinar frecuencia y condicion de oscilacion
- Analizar osciladores sinusoidales: puente de Wien, desplazamiento de fase, Colpitts y Hartley
- Analizar osciladores de relajacion: comparador con histeresis y temporizador 555 en modo astable
- Resolver ejercicios de diseno: calcular componentes para una frecuencia objetivo

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
from matplotlib.lines import Line2D
import schemdraw
import schemdraw.elements as elm

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.labelsize'] = 13
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['figure.dpi'] = 100

COLOR_PRINCIPAL = '#2171b5'   # azul - curvas principales
COLOR_RECTA = '#cb181d'       # rojo - limites, umbrales
COLOR_PUNTO = '#238b45'       # verde - puntos de operacion, resultados OK
COLOR_SEC = '#ff7f0e'         # naranja - curvas secundarias
COLOR_FILL = '#a6cee3'        # azul claro - rellenos
COLOR_FILL2 = '#b2df8a'       # verde claro - rellenos secundarios

print('Configuracion lista.')

---

## 1. Introduccion: que es un oscilador?

Un **oscilador electronico** es un circuito que genera una senal periodica **sin necesidad de senal de entrada**. Es, en esencia, un **amplificador inestable**: la salida se realimenta a la entrada de forma que el sistema se autoexcita.

### Analogia intuitiva

Imagina un microfono acoplado a un altavoz. Si acercas el microfono al altavoz, el sonido se realimenta y aparece un pitido agudo (acople). Eso es una oscilacion: el sistema amplifica su propia senal una y otra vez.

### Clasificacion general

| Tipo | Forma de onda | Mecanismo | Ejemplos |
|------|--------------|-----------|----------|
| **Lineal (sinusoidal)** | Senoidal | Realimentacion positiva selectiva en frecuencia | Wien, Colpitts, Hartley, cristal |
| **No lineal (relajacion)** | Cuadrada, triangular | Conmutacion entre dos estados | Comparador + RC, 555 astable |

**Concepto clave:** Todo oscilador necesita un **elemento que seleccione la frecuencia** (red RC, red LC o cristal) y un **elemento activo** que compense las perdidas (amplificador, transistor, OPAM).

---

## 2. Criterio de Barkhausen

El **criterio de Barkhausen** establece las condiciones necesarias para que un circuito con realimentacion oscile de forma sostenida.

Consideremos un amplificador con ganancia $A(j\omega)$ y una red de realimentacion $\beta(j\omega)$. La **ganancia de lazo** es:

$$T(j\omega) = A(j\omega) \cdot \beta(j\omega)$$

Para que exista oscilacion sostenida a la frecuencia $\omega_0$:

$$\boxed{|A(j\omega_0) \cdot \beta(j\omega_0)| = 1 \qquad \text{(condicion de magnitud)}}$$

$$\boxed{\angle A(j\omega_0) \cdot \beta(j\omega_0) = 0^\circ \;(\text{o }360^\circ) \qquad \text{(condicion de fase)}}$$

**Interpretacion practica:**

- Si $|T| < 1$ $\to$ la senal se atenua en cada vuelta $\to$ **las oscilaciones se amortiguan**
- Si $|T| = 1$ $\to$ la senal se mantiene $\to$ **oscilacion sostenida**
- Si $|T| > 1$ $\to$ la senal crece $\to$ **oscilacion creciente** (hasta que la no linealidad del amplificador la limita)

> **Nota de diseno:** En la practica se diseña con $|T|$ ligeramente mayor que 1 para garantizar el arranque. La amplitud se estabiliza por la saturacion del amplificador o por un mecanismo de control automatico de ganancia (AGC).

In [ ]:
# Diagrama de bloques: realimentacion positiva (oscilador)
fig, ax = plt.subplots(figsize=(10, 5))
ax.set_xlim(0, 10)
ax.set_ylim(0, 6)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('Diagrama de bloques de un oscilador con realimentacion', fontsize=14, fontweight='bold')

# Sumador
circle = plt.Circle((2, 3.5), 0.4, fill=False, color='black', lw=2)
ax.add_patch(circle)
ax.text(2, 3.5, '+', fontsize=16, ha='center', va='center', fontweight='bold')

# Bloque A(jw)
rect_a = FancyBboxPatch((3.5, 2.8), 2, 1.4, boxstyle='round,pad=0.1',
                         facecolor=COLOR_FILL, edgecolor=COLOR_PRINCIPAL, lw=2)
ax.add_patch(rect_a)
ax.text(4.5, 3.5, r'$A(j\omega)$', fontsize=16, ha='center', va='center', fontweight='bold')

# Bloque beta(jw)
rect_b = FancyBboxPatch((3.5, 0.6), 2, 1.2, boxstyle='round,pad=0.1',
                         facecolor=COLOR_FILL2, edgecolor=COLOR_PUNTO, lw=2)
ax.add_patch(rect_b)
ax.text(4.5, 1.2, r'$\beta(j\omega)$', fontsize=16, ha='center', va='center', fontweight='bold')

# Flechas
ax.annotate('', xy=(3.5, 3.5), xytext=(2.4, 3.5),
            arrowprops=dict(arrowstyle='->', lw=2, color='black'))
ax.annotate('', xy=(7.5, 3.5), xytext=(5.5, 3.5),
            arrowprops=dict(arrowstyle='->', lw=2, color='black'))

# Salida
ax.annotate('', xy=(8.5, 3.5), xytext=(7.5, 3.5),
            arrowprops=dict(arrowstyle='->', lw=2, color='black'))
ax.text(8.7, 3.5, r'$V_o$', fontsize=14, ha='left', va='center')

# Conexion salida -> beta
ax.plot([7.5, 7.5], [3.5, 1.2], 'k-', lw=2)
ax.annotate('', xy=(5.5, 1.2), xytext=(7.5, 1.2),
            arrowprops=dict(arrowstyle='<-', lw=2, color='black'))

# Conexion beta -> sumador
ax.plot([3.5, 1.2], [1.2, 1.2], 'k-', lw=2)
ax.plot([1.2, 1.2], [1.2, 3.1], 'k-', lw=2)
ax.annotate('', xy=(1.6, 3.5), xytext=(0.5, 3.5),
            arrowprops=dict(arrowstyle='->', lw=2, color='black'))

# Etiquetas
ax.text(6.5, 3.8, r'$V_o$', fontsize=12, ha='center')
ax.text(6.5, 1.5, r'$V_f = \beta \cdot V_o$', fontsize=12, ha='center', color=COLOR_PUNTO)
ax.text(0.3, 3.8, r'$V_i = 0$', fontsize=12, ha='left', color=COLOR_RECTA)

# Condicion de Barkhausen
ax.text(5, 5.3, r'Barkhausen: $|A \cdot \beta| = 1$ y $\angle(A \cdot \beta) = 0^\circ$',
        fontsize=13, ha='center', va='center',
        bbox=dict(boxstyle='round', facecolor='lightyellow', edgecolor=COLOR_SEC, lw=2))

plt.tight_layout()
plt.show()

In [ ]:
# Diagrama de Bode mostrando el criterio de Barkhausen
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
fig.suptitle('Diagrama de Bode de la ganancia de lazo T(jw)', fontsize=14, fontweight='bold')

# Frecuencias
f = np.logspace(1, 6, 1000)
w = 2 * np.pi * f

# Modelo simplificado de ganancia de lazo (paso-banda)
f0 = 1e4  # frecuencia de oscilacion
Q = 10
w0 = 2 * np.pi * f0
T_mag = 1.0 / np.sqrt(1 + Q**2 * (f/f0 - f0/f)**2)
T_phase = -np.arctan2(Q * (f/f0 - f0/f), 1) * 180 / np.pi

# Magnitud
ax1.semilogx(f, 20*np.log10(T_mag), color=COLOR_PRINCIPAL, lw=2.5, label=r'$|T(j\omega)|$ (dB)')
ax1.axhline(y=0, color=COLOR_RECTA, ls='--', lw=2, label='0 dB (|T|=1)')
ax1.axvline(x=f0, color=COLOR_PUNTO, ls=':', lw=1.5, alpha=0.7)
ax1.plot(f0, 0, 'o', color=COLOR_PUNTO, ms=14, zorder=5, label=r'$f_0$ = Barkhausen')
ax1.set_ylabel('Magnitud (dB)')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)
ax1.set_ylim(-30, 5)
ax1.annotate(f'$f_0 = {f0/1e3:.0f}$ kHz\n|T| = 1 (0 dB)',
             xy=(f0, 0), xytext=(f0*5, -8),
             fontsize=12, color=COLOR_PUNTO, fontweight='bold',
             arrowprops=dict(arrowstyle='->', color=COLOR_PUNTO, lw=2),
             bbox=dict(boxstyle='round', facecolor='white', edgecolor=COLOR_PUNTO))

# Fase
ax2.semilogx(f, T_phase, color=COLOR_SEC, lw=2.5, label=r'$\angle T(j\omega)$')
ax2.axhline(y=0, color=COLOR_RECTA, ls='--', lw=2, label=r'$0^\circ$')
ax2.axvline(x=f0, color=COLOR_PUNTO, ls=':', lw=1.5, alpha=0.7)
ax2.plot(f0, 0, 'o', color=COLOR_PUNTO, ms=14, zorder=5)
ax2.set_xlabel('Frecuencia (Hz)')
ax2.set_ylabel('Fase (grados)')
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)
ax2.set_ylim(-100, 100)

plt.tight_layout()
plt.show()

---

## 3. Formulario completo

### 3.1 Osciladores sinusoidales

| Oscilador | Frecuencia de oscilacion | Condicion de ganancia | Red selectiva |
|-----------|-------------------------|----------------------|---------------|
| **Puente de Wien** | $f_0 = \dfrac{1}{2\pi RC}$ | $A = 3$ (no inversora) | RC serie + RC paralelo |
| **Desplazamiento de fase** | $f_0 = \dfrac{1}{2\pi RC\sqrt{6}}$ | $A = 29$ (3 etapas RC) | 3 secciones RC |
| **Colpitts** | $f_0 = \dfrac{1}{2\pi\sqrt{L \cdot \dfrac{C_1 C_2}{C_1 + C_2}}}$ | $g_m R > \dfrac{C_1}{C_2}$ | L + C1 + C2 |
| **Hartley** | $f_0 = \dfrac{1}{2\pi\sqrt{(L_1 + L_2) \cdot C}}$ | $g_m R > \dfrac{L_2}{L_1}$ | L1 + L2 + C |
| **Cristal** | $f_0 \approx \dfrac{1}{2\pi\sqrt{L_s C_s}}$ | Determinada por el cristal | Cristal piezoelectrico |

### 3.2 Osciladores de relajacion

| Oscilador | Periodo / Frecuencia | Parametros clave |
|-----------|---------------------|------------------|
| **Comparador + RC** | $T = 2RC \ln\left(\dfrac{1+\beta}{1-\beta}\right)$ | $\beta = \dfrac{R_1}{R_1 + R_2}$ |
| **555 astable** | $T = 0.693(R_A + 2R_B)C$ | $f = \dfrac{1.44}{(R_A + 2R_B)C}$ |

---

## 4. Osciladores sinusoidales

### 4.1 Oscilador de puente de Wien

El puente de Wien es el oscilador sinusoidal mas utilizado en el rango de audiofrecuencias (20 Hz - 20 kHz). Usa un **amplificador no inversor** con una red RC que proporciona realimentacion positiva.

**Estructura:**
- Amplificador no inversor con ganancia $A = 1 + R_f/R_g$
- Red de realimentacion: RC serie + RC paralelo (puente de Wien)

**Funcion de transferencia de la red:**

$$\beta(j\omega) = \frac{1}{3 + j\left(\dfrac{\omega}{\omega_0} - \dfrac{\omega_0}{\omega}\right)}$$

A la frecuencia $\omega_0 = 1/(RC)$, el termino imaginario se anula y $\beta = 1/3$.

$$\boxed{f_0 = \frac{1}{2\pi RC}} \qquad \boxed{A = 3 \implies R_f = 2R_g}$$

**Como afectan los componentes:**
- Si **$R$ o $C$ aumentan** $\to$ $f_0$ disminuye (oscilacion mas lenta)
- Si **$R_f/R_g > 2$** $\to$ $A > 3$ $\to$ oscilacion creciente (distorsion)
- Si **$R_f/R_g < 2$** $\to$ $A < 3$ $\to$ oscilaciones se amortiguan

**Ventaja:** Frecuencia facilmente ajustable con un potenciometro dual (R variable).

**Truco para el examen:** En el puente de Wien, SIEMPRE $A = 3$, es decir, $R_f = 2R_g$.

In [ ]:
# Circuito del oscilador de puente de Wien
fig, ax = plt.subplots(figsize=(8, 7))
ax.set_title('Oscilador de puente de Wien', fontsize=14, fontweight='bold')

d = schemdraw.Drawing(canvas=ax)

# OPAM
opam = d.add(elm.Opamp().right().anchor('in1'))

# Red de realimentacion negativa (Rf y Rg)
d.add(elm.Line().at(opam.in2).left().length(1))
rg_start = d.here
d.add(elm.Resistor().down().label(r'$R_g$', loc='right').length(3))
d.add(elm.Ground())
d.add(elm.Line().at(rg_start).left().length(0.5))
rf_junc = d.here
d.add(elm.Line().up().length(2))
d.add(elm.Resistor().right().label(r'$R_f = 2R_g$', loc='top').length(4))
d.add(elm.Line().down().length(2))
d.add(elm.Line().right().tox(opam.out))
d.add(elm.Dot())

# Salida
d.add(elm.Line().at(opam.out).right().length(1.5))
d.add(elm.Dot(open=True).label(r'$V_o$', loc='right'))

# Red de Wien (realimentacion positiva)
d.add(elm.Line().at(opam.in1).left().length(1))
wien_top = d.here
d.add(elm.Resistor().left().label(r'$R$', loc='top').length(2))
d.add(elm.Capacitor().left().label(r'$C$', loc='top').length(2))
d.add(elm.Line().down().length(2))
d.add(elm.Line().right().length(2))
wien_mid = d.here
d.add(elm.Line().right().length(2))
d.add(elm.Line().down().length(1.5))
d.add(elm.Ground())

d.add(elm.Line().at(wien_mid).up().length(0.01))
d.add(elm.Resistor().up().label(r'$R$', loc='left').length(2))
d.add(elm.Line().at(wien_mid).down().length(0.01))
d.add(elm.Capacitor().down().label(r'$C$', loc='left').length(2))

d.draw()
plt.tight_layout()
plt.show()

### 4.2 Oscilador de desplazamiento de fase

Usa un amplificador inversor ($180^\circ$ de desfase) y una red de **3 secciones RC** que aporta otros $180^\circ$ a la frecuencia de oscilacion, completando los $360^\circ$ necesarios.

$$\boxed{f_0 = \frac{1}{2\pi RC\sqrt{6}}} \qquad \boxed{A = 29 \implies R_f = 29R}$$

**Cada etapa RC aporta $60^\circ$** a la frecuencia $f_0$ (3 etapas $\times$ $60^\circ$ = $180^\circ$).

**Como afectan los componentes:**
- Si **$R$ o $C$ aumentan** $\to$ $f_0$ disminuye
- Si **la ganancia $< 29$** $\to$ no arranca la oscilacion
- Cada etapa RC introduce atenuacion: por eso se necesita $A = 29$ (compensar la atenuacion total de $1/29$)

**Desventaja:** La ganancia minima es alta ($A = 29$), lo que puede generar distorsion. Ademas, cambiar la frecuencia requiere ajustar 3 resistencias o 3 condensadores simultaneamente.

### 4.3 Osciladores LC: Colpitts y Hartley

Estos osciladores usan un **tanque LC** como red selectora de frecuencia. Son ideales para radiofrecuencias (RF).

#### Colpitts: divisor capacitivo

El tanque esta formado por $L$ en paralelo con $C_1$ y $C_2$ en serie:

$$\boxed{f_0 = \frac{1}{2\pi\sqrt{L \cdot \dfrac{C_1 C_2}{C_1 + C_2}}}}$$

**Condicion de oscilacion:** $g_m R > C_1/C_2$

La realimentacion se toma del divisor capacitivo ($C_1$ y $C_2$).

#### Hartley: divisor inductivo

El tanque esta formado por $C$ en paralelo con $L_1$ y $L_2$ en serie:

$$\boxed{f_0 = \frac{1}{2\pi\sqrt{(L_1 + L_2) \cdot C}}}$$

**Condicion de oscilacion:** $g_m R > L_2/L_1$

**Comparativa Colpitts vs Hartley:**

| Caracteristica | Colpitts | Hartley |
|----------------|----------|----------|
| Divisor | Capacitivo ($C_1$, $C_2$) | Inductivo ($L_1$, $L_2$) |
| Ventaja | Menos ruido, mejor estabilidad | Facil ajuste de frecuencia |
| Uso tipico | VHF, UHF | HF, VHF |
| Formula $f_0$ | Usa $C_{eq} = C_1 C_2/(C_1+C_2)$ | Usa $L_{eq} = L_1 + L_2$ |

---

## 5. Osciladores de relajacion

### 5.1 Comparador con histeresis (astable)

Un **comparador con realimentacion positiva** (histeresis) combinado con una red RC genera una onda cuadrada. El condensador se carga y descarga entre los **umbrales de conmutacion** $V_{TH}$ y $V_{TL}$.

$$\beta = \frac{R_1}{R_1 + R_2}$$

$$V_{TH} = +\beta V_{sat} \qquad V_{TL} = -\beta V_{sat}$$

$$\boxed{T = 2RC \ln\left(\frac{1 + \beta}{1 - \beta}\right)}$$

Donde $V_{sat}$ es la tension de saturacion del comparador.

### 5.2 Temporizador 555 en modo astable

El **555 en modo astable** genera una onda cuadrada con periodo ajustable. El condensador se carga a traves de $R_A + R_B$ y se descarga a traves de $R_B$.

$$\boxed{T = 0.693(R_A + 2R_B)C} \qquad \boxed{f = \frac{1.44}{(R_A + 2R_B)C}}$$

**Tiempos de carga y descarga:**

$$t_{HIGH} = 0.693(R_A + R_B)C \qquad t_{LOW} = 0.693 \cdot R_B \cdot C$$

**Ciclo de trabajo:**

$$D = \frac{t_{HIGH}}{T} = \frac{R_A + R_B}{R_A + 2R_B}$$

**Como afectan los componentes:**
- Si **$R_A$ aumenta** $\to$ $t_{HIGH}$ aumenta $\to$ $D$ aumenta (mas tiempo en alto)
- Si **$R_B$ aumenta** $\to$ tanto $t_{HIGH}$ como $t_{LOW}$ aumentan, pero $D \to 50\%$
- Si **$C$ aumenta** $\to$ el periodo $T$ aumenta (frecuencia menor)

**Nota:** Con el 555 basico, siempre $D > 50\%$ porque $t_{HIGH} > t_{LOW}$.

In [ ]:
# Circuito del temporizador 555 en modo astable
fig, ax = plt.subplots(figsize=(8, 8))
ax.set_title('Temporizador 555 en modo astable', fontsize=14, fontweight='bold')

d = schemdraw.Drawing(canvas=ax)

# VCC
d.add(elm.Line().right().length(0))
start = d.here
d.add(elm.Line().up().length(5))
top = d.here
d.add(elm.Dot(open=True).label(r'$V_{CC}$', loc='top'))

# RA desde VCC
d.add(elm.Line().at(top).right().length(2))
d.add(elm.Resistor().down().label(r'$R_A$', loc='right').length(2.5))
ra_bottom = d.here

# RB desde RA
d.add(elm.Resistor().down().label(r'$R_B$', loc='right').length(2.5))
rb_bottom = d.here

# Condensador
d.add(elm.Capacitor().down().label(r'$C$', loc='right').length(2.5))
d.add(elm.Ground())

# Bloque 555 (caja)
box_x = 5.5
box_y = 1.0
box_w = 3.0
box_h = 5.0
rect_555 = FancyBboxPatch((box_x, box_y), box_w, box_h,
                           boxstyle='round,pad=0.1',
                           facecolor=COLOR_FILL, edgecolor=COLOR_PRINCIPAL, lw=2.5)
ax.add_patch(rect_555)
ax.text(box_x + box_w/2, box_y + box_h/2, '555', fontsize=20,
        ha='center', va='center', fontweight='bold', color=COLOR_PRINCIPAL)

# Pines del 555 (etiquetas)
ax.text(box_x - 0.3, box_y + 4.0, 'VCC (8)', fontsize=9, ha='right', va='center')
ax.text(box_x - 0.3, box_y + 3.0, 'TRIG (2)', fontsize=9, ha='right', va='center')
ax.text(box_x - 0.3, box_y + 2.0, 'THRESH (6)', fontsize=9, ha='right', va='center')
ax.text(box_x + box_w + 0.3, box_y + 3.0, 'OUT (3)', fontsize=9, ha='left', va='center')
ax.text(box_x + box_w + 0.3, box_y + 2.0, 'DISCH (7)', fontsize=9, ha='left', va='center')
ax.text(box_x + box_w/2, box_y - 0.4, 'GND (1)', fontsize=9, ha='center', va='top')

# VCC -> pin 8
ax.annotate('', xy=(box_x, box_y+4.0), xytext=(box_x-1.5, box_y+4.0),
            arrowprops=dict(arrowstyle='-', lw=1.5, color='black'))

# Salida
ax.annotate('', xy=(box_x+box_w+2, box_y+3.0), xytext=(box_x+box_w, box_y+3.0),
            arrowprops=dict(arrowstyle='->', lw=2, color=COLOR_PUNTO))
ax.text(box_x+box_w+2.2, box_y+3.0, r'$V_o$', fontsize=14, ha='left', va='center', color=COLOR_PUNTO)

# Formulas
ax.text(box_x + box_w/2, box_y - 1.5,
        r'$T = 0.693(R_A + 2R_B)C$' + '\n' + r'$f = \frac{1.44}{(R_A + 2R_B)C}$',
        fontsize=12, ha='center', va='center',
        bbox=dict(boxstyle='round', facecolor='lightyellow', edgecolor=COLOR_SEC, lw=1.5))

d.draw()
plt.tight_layout()
plt.show()

In [ ]:
# Formas de onda: sinusoidal vs cuadrada vs triangular
fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
fig.suptitle('Formas de onda de osciladores', fontsize=14, fontweight='bold')

t = np.linspace(0, 4e-3, 2000)
f0 = 1e3  # 1 kHz

# Sinusoidal
v_sin = 5 * np.sin(2 * np.pi * f0 * t)
axes[0].plot(t*1e3, v_sin, color=COLOR_PRINCIPAL, lw=2.5)
axes[0].set_ylabel('Tension (V)')
axes[0].set_title('Sinusoidal (Wien, Colpitts, Hartley)', fontsize=12)
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(-7, 7)
axes[0].axhline(y=0, color='gray', ls='-', lw=0.5)

# Cuadrada
from scipy import signal as sig
v_sq = 5 * sig.square(2 * np.pi * f0 * t)
axes[1].plot(t*1e3, v_sq, color=COLOR_RECTA, lw=2.5)
axes[1].set_ylabel('Tension (V)')
axes[1].set_title('Cuadrada (555 astable, comparador)', fontsize=12)
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(-7, 7)
axes[1].axhline(y=0, color='gray', ls='-', lw=0.5)

# Triangular
v_tri = 5 * sig.sawtooth(2 * np.pi * f0 * t, width=0.5)
axes[2].plot(t*1e3, v_tri, color=COLOR_SEC, lw=2.5)
axes[2].set_ylabel('Tension (V)')
axes[2].set_title('Triangular (integrador + comparador)', fontsize=12)
axes[2].set_xlabel('Tiempo (ms)')
axes[2].grid(True, alpha=0.3)
axes[2].set_ylim(-7, 7)
axes[2].axhline(y=0, color='gray', ls='-', lw=0.5)

plt.tight_layout()
plt.show()

---

## 6. Metodologia de resolucion

### Pasos para analizar/disenar un oscilador

**Paso 1:** Identificar el **tipo** de oscilador (Wien, Colpitts, 555, etc.)

**Paso 2:** Escribir la **formula de frecuencia** correspondiente

**Paso 3:** Verificar la **condicion de ganancia** (Barkhausen)

**Paso 4:** Calcular la frecuencia de oscilacion a partir de los componentes, o calcular los componentes a partir de la frecuencia deseada

**Paso 5:** Para osciladores de relajacion, calcular tambien el **periodo**, el **ciclo de trabajo** y los **tiempos de carga/descarga**

**Errores frecuentes:**
- Olvidar que en el puente de Wien la ganancia debe ser **exactamente 3** ($R_f = 2R_g$)
- Confundir Colpitts (divisor capacitivo) con Hartley (divisor inductivo)
- En el 555, usar $R_A + R_B$ en vez de $R_A + 2R_B$ para el periodo total
- Olvidar el factor $\sqrt{6}$ en el oscilador de desplazamiento de fase

---

## 7. Ejercicios resueltos

#### Ejercicio resuelto 1: Puente de Wien

**Datos:** $R = 10$ k$\Omega$, $C = 10$ nF, $R_g = 10$ k$\Omega$

**Paso 1:** Oscilador de puente de Wien $\to$ $f_0 = \dfrac{1}{2\pi RC}$, condicion $A = 3$

**Paso 2:** Frecuencia de oscilacion:

$$f_0 = \frac{1}{2\pi \times 10\text{k}\Omega \times 10\text{nF}} = \frac{1}{2\pi \times 10^4 \times 10^{-8}} = \frac{1}{2\pi \times 10^{-4}}$$

$$\boxed{f_0 = 1591.5\;\text{Hz} \approx 1.59\;\text{kHz}}$$

**Paso 3:** Condicion de ganancia:

$$A = 3 \implies 1 + \frac{R_f}{R_g} = 3 \implies R_f = 2R_g = 2 \times 10\text{k}\Omega$$

$$\boxed{R_f = 20\;\text{k}\Omega}$$

**Verificacion:** $A = 1 + 20/10 = 3$ $\checkmark$ $\to$ **Oscilacion sostenida a 1.59 kHz.**

#### Ejercicio resuelto 2: Oscilador de desplazamiento de fase

**Datos:** Se desea $f_0 = 5$ kHz, $C = 1$ nF. Calcular $R$ y $R_f$.

**Paso 1:** Oscilador de desplazamiento de fase, 3 etapas RC.

**Paso 2:** Despejar $R$ de la formula:

$$f_0 = \frac{1}{2\pi RC\sqrt{6}} \implies R = \frac{1}{2\pi f_0 C \sqrt{6}}$$

$$R = \frac{1}{2\pi \times 5000 \times 10^{-9} \times \sqrt{6}} = \frac{1}{2\pi \times 5 \times 10^{-6} \times 2.449}$$

$$R = \frac{1}{7.695 \times 10^{-5}} = 12995\;\Omega$$

$$\boxed{R \approx 13\;\text{k}\Omega}$$

**Paso 3:** Condicion de ganancia:

$$A = 29 \implies R_f = 29R = 29 \times 13\text{k}\Omega$$

$$\boxed{R_f = 377\;\text{k}\Omega}$$

**Verificacion:** Con $R = 13$ k$\Omega$ y $C = 1$ nF:

$$f_0 = \frac{1}{2\pi \times 13000 \times 10^{-9} \times 2.449} = \frac{1}{1.999 \times 10^{-4}} = 5003\;\text{Hz} \approx 5\;\text{kHz}\;\checkmark$$

#### Ejercicio resuelto 3: Colpitts

**Datos:** $L = 100\;\mu$H, $C_1 = 100$ pF, $C_2 = 400$ pF

**Paso 1:** Oscilador Colpitts $\to$ $f_0 = \dfrac{1}{2\pi\sqrt{L \cdot C_{eq}}}$

**Paso 2:** Capacidad equivalente serie:

$$C_{eq} = \frac{C_1 C_2}{C_1 + C_2} = \frac{100 \times 400}{100 + 400} = \frac{40000}{500} = 80\;\text{pF}$$

**Paso 3:** Frecuencia de oscilacion:

$$f_0 = \frac{1}{2\pi\sqrt{100 \times 10^{-6} \times 80 \times 10^{-12}}} = \frac{1}{2\pi\sqrt{8 \times 10^{-15}}}$$

$$f_0 = \frac{1}{2\pi \times 8.944 \times 10^{-8}} = \frac{1}{5.621 \times 10^{-7}}$$

$$\boxed{f_0 = 1.779\;\text{MHz} \approx 1.78\;\text{MHz}}$$

**Paso 4:** Condicion de oscilacion:

$$g_m R > \frac{C_1}{C_2} = \frac{100}{400} = 0.25$$

El producto $g_m R$ debe ser mayor que 0.25 para garantizar el arranque.

#### Ejercicio resuelto 4: 555 astable

**Datos:** $R_A = 4.7$ k$\Omega$, $R_B = 10$ k$\Omega$, $C = 100$ nF

**Paso 1:** 555 en modo astable.

**Paso 2:** Periodo y frecuencia:

$$T = 0.693(R_A + 2R_B)C = 0.693 \times (4700 + 2 \times 10000) \times 100 \times 10^{-9}$$

$$T = 0.693 \times 24700 \times 10^{-7} = 0.693 \times 2.47 \times 10^{-3}$$

$$\boxed{T = 1.712\;\text{ms}}$$

$$\boxed{f = \frac{1}{T} = \frac{1}{1.712 \times 10^{-3}} = 584.1\;\text{Hz}}$$

**Paso 3:** Tiempos de carga y descarga:

$$t_{HIGH} = 0.693(R_A + R_B)C = 0.693 \times 14700 \times 10^{-7} = 1.019\;\text{ms}$$

$$t_{LOW} = 0.693 \cdot R_B \cdot C = 0.693 \times 10000 \times 10^{-7} = 0.693\;\text{ms}$$

**Paso 4:** Ciclo de trabajo:

$$D = \frac{R_A + R_B}{R_A + 2R_B} = \frac{14700}{24700} = 0.595$$

$$\boxed{D = 59.5\%}$$

**Verificacion:** $t_{HIGH} + t_{LOW} = 1.019 + 0.693 = 1.712$ ms $= T$ $\checkmark$. $D > 50\%$ como se espera del 555 basico $\checkmark$.

---

## 8. Catalogo completo de ejercicios: todos los patrones

| # | Tipo | Componentes | Ecuacion clave | Dificultad |
|---|------|-------------|----------------|------------|
| 1 | Puente de Wien | R, C, OPAM | $f_0 = \dfrac{1}{2\pi RC}$ | Baja |
| 2 | Desplazamiento de fase | R, C, OPAM | $f_0 = \dfrac{1}{2\pi RC\sqrt{6}}$ | Media |
| 3 | Colpitts | L, C1, C2, transistor | $f_0 = \dfrac{1}{2\pi\sqrt{LC_{eq}}}$ | Media |
| 4 | Hartley | L1, L2, C, transistor | $f_0 = \dfrac{1}{2\pi\sqrt{L_{eq}C}}$ | Media |
| 5 | 555 astable | RA, RB, C | $f = \dfrac{1.44}{(R_A+2R_B)C}$ | Baja |

### 8.1 Tipo 1: Puente de Wien

$$\boxed{f_0 = \frac{1}{2\pi RC}} \qquad A = 3 \implies R_f = 2R_g$$

**Variantes de problema:**
- Dados $R$ y $C$, calcular $f_0$ y $R_f$
- Dada $f_0$ y $C$, calcular $R$ y $R_f$
- Dado el circuito completo, verificar si oscila y a que frecuencia

**Clave:** La ganancia es SIEMPRE 3. Si te dan una ganancia diferente, el circuito no oscilara de forma sinusoidal pura.

#### Ejercicio catalogo: Wien inverso

**Datos:** $f_0 = 10$ kHz, $C = 4.7$ nF. Calcular $R$, $R_f$ (con $R_g = 10$ k$\Omega$).

$$R = \frac{1}{2\pi f_0 C} = \frac{1}{2\pi \times 10^4 \times 4.7 \times 10^{-9}} = \frac{1}{2.953 \times 10^{-4}} = 3386\;\Omega$$

$$\boxed{R \approx 3.4\;\text{k}\Omega} \qquad R_f = 2 \times 10\text{k} = 20\;\text{k}\Omega$$

### 8.2 Tipo 2: Desplazamiento de fase

$$\boxed{f_0 = \frac{1}{2\pi RC\sqrt{6}}} \qquad A = 29 \implies R_f = 29R$$

**Variantes de problema:**
- Dados $R$ y $C$, calcular $f_0$
- Dada $f_0$, disenar la red RC
- Verificar si la ganancia es suficiente para oscilar

**Clave:** El factor $\sqrt{6} \approx 2.449$ es facil de olvidar. La ganancia minima de 29 es mucho mayor que la del Wien (3).

#### Ejercicio catalogo: Verificar oscilacion

**Datos:** $R = 22$ k$\Omega$, $C = 2.2$ nF, $R_f = 500$ k$\Omega$, $R_i = 22$ k$\Omega$ (amplificador inversor).

$$f_0 = \frac{1}{2\pi \times 22000 \times 2.2 \times 10^{-9} \times 2.449} = \frac{1}{7.436 \times 10^{-4}} = 1344.8\;\text{Hz}$$

$$A = \frac{R_f}{R_i} = \frac{500}{22} = 22.7$$

Como $A = 22.7 < 29$ $\to$ **NO oscila.** La ganancia es insuficiente.

### 8.3 Tipo 3: Colpitts

$$\boxed{f_0 = \frac{1}{2\pi\sqrt{L \cdot \frac{C_1 C_2}{C_1 + C_2}}}} \qquad g_m R > \frac{C_1}{C_2}$$

**Variantes de problema:**
- Dados $L$, $C_1$, $C_2$, calcular $f_0$
- Dada $f_0$ y la relacion $C_1/C_2$, disenar el circuito
- Verificar la condicion de oscilacion con $g_m$ y $R$ dados

**Clave:** Primero calcular $C_{eq} = C_1 C_2/(C_1 + C_2)$, despues usar $f_0 = 1/(2\pi\sqrt{LC_{eq}})$.

#### Ejercicio catalogo: Disenar para $f_0$ dada

**Datos:** $f_0 = 5$ MHz, $C_1 = C_2 = C$ (iguales), $L = 10\;\mu$H. Calcular $C$.

$$C_{eq} = \frac{C \cdot C}{C + C} = \frac{C}{2}$$

$$f_0 = \frac{1}{2\pi\sqrt{L \cdot C/2}} \implies C = \frac{2}{(2\pi f_0)^2 L}$$

$$C = \frac{2}{(2\pi \times 5 \times 10^6)^2 \times 10 \times 10^{-6}} = \frac{2}{9.870 \times 10^{15} \times 10^{-5}} = \frac{2}{9.870 \times 10^{10}}$$

$$\boxed{C = 20.3\;\text{pF}}$$

### 8.4 Tipo 4: Hartley

$$\boxed{f_0 = \frac{1}{2\pi\sqrt{(L_1 + L_2) \cdot C}}} \qquad g_m R > \frac{L_2}{L_1}$$

**Variantes de problema:**
- Dados $L_1$, $L_2$, $C$, calcular $f_0$
- Disenar para una frecuencia con relacion $L_1/L_2$ dada

**Clave:** En Hartley se suman las inductancias ($L_{eq} = L_1 + L_2$), mientras que en Colpitts se calculan capacidades en serie.

#### Ejercicio catalogo: Hartley directo

**Datos:** $L_1 = 50\;\mu$H, $L_2 = 150\;\mu$H, $C = 100$ pF.

$$L_{eq} = L_1 + L_2 = 50 + 150 = 200\;\mu\text{H}$$

$$f_0 = \frac{1}{2\pi\sqrt{200 \times 10^{-6} \times 100 \times 10^{-12}}} = \frac{1}{2\pi\sqrt{2 \times 10^{-14}}}$$

$$f_0 = \frac{1}{2\pi \times 1.414 \times 10^{-7}} = \frac{1}{8.886 \times 10^{-7}}$$

$$\boxed{f_0 = 1.125\;\text{MHz}}$$

**Condicion:** $g_m R > L_2/L_1 = 150/50 = 3$

### 8.5 Tipo 5: 555 astable

$$\boxed{T = 0.693(R_A + 2R_B)C} \qquad \boxed{f = \frac{1.44}{(R_A + 2R_B)C}}$$

$$D = \frac{R_A + R_B}{R_A + 2R_B} \qquad t_{HIGH} = 0.693(R_A + R_B)C \qquad t_{LOW} = 0.693 R_B C$$

**Variantes de problema:**
- Dados $R_A$, $R_B$, $C$, calcular $f$, $T$, $D$
- Dada $f$ y $D$, disenar $R_A$, $R_B$, $C$
- Calcular $R_A$ y $R_B$ para un ciclo de trabajo especifico

**Clave:** $R_A$ solo afecta a la carga, $R_B$ afecta a carga Y descarga.

#### Ejercicio catalogo: Disenar para $f$ y $D$

**Datos:** $f = 1$ kHz, $D = 60\%$, $C = 100$ nF.

**Paso 1:** Del ciclo de trabajo:

$$D = \frac{R_A + R_B}{R_A + 2R_B} = 0.6$$

$$R_A + R_B = 0.6(R_A + 2R_B) = 0.6R_A + 1.2R_B$$

$$0.4R_A = 0.2R_B \implies R_A = 0.5R_B$$

**Paso 2:** De la frecuencia:

$$f = \frac{1.44}{(R_A + 2R_B)C} \implies R_A + 2R_B = \frac{1.44}{fC} = \frac{1.44}{10^3 \times 10^{-7}} = 14400\;\Omega$$

$$0.5R_B + 2R_B = 14400 \implies 2.5R_B = 14400$$

$$\boxed{R_B = 5.76\;\text{k}\Omega} \qquad \boxed{R_A = 2.88\;\text{k}\Omega}$$

**Verificacion:** $T = 0.693 \times (2880 + 2 \times 5760) \times 10^{-7} = 0.693 \times 14400 \times 10^{-7} = 0.998$ ms $\to$ $f = 1002$ Hz $\approx 1$ kHz $\checkmark$

### Tabla resumen de formulas por tipo de oscilador

| Tipo | Formula de $f_0$ | Condicion de ganancia | Red selectiva |
|------|-----------------|----------------------|---------------|
| 1. Wien | $\dfrac{1}{2\pi RC}$ | $A = 3$ | RC serie + RC paralelo |
| 2. Desplaz. fase | $\dfrac{1}{2\pi RC\sqrt{6}}$ | $A = 29$ | 3 etapas RC |
| 3. Colpitts | $\dfrac{1}{2\pi\sqrt{LC_{eq}}}$ | $g_m R > C_1/C_2$ | L + $C_1$ serie $C_2$ |
| 4. Hartley | $\dfrac{1}{2\pi\sqrt{L_{eq}C}}$ | $g_m R > L_2/L_1$ | $L_1 + L_2$ + C |
| 5. 555 astable | $\dfrac{1.44}{(R_A+2R_B)C}$ | N/A (digital) | RC + comparadores internos |

---

## 9. Resumen y tabla de formulas clave

### Conceptos esenciales

1. **Oscilador = amplificador inestable** con realimentacion positiva
2. **Criterio de Barkhausen:** $|A\beta| = 1$ y $\angle(A\beta) = 0^\circ$ a la frecuencia de oscilacion
3. Los **osciladores sinusoidales** (Wien, Colpitts, Hartley) usan redes selectivas en frecuencia
4. Los **osciladores de relajacion** (555, comparador) conmutan entre dos estados
5. En la practica, se diseña con $|A\beta| > 1$ para garantizar el arranque

### Formulas clave para el examen

| Formula | Oscilador | Cuando usarla |
|---------|-----------|---------------|
| $f_0 = \dfrac{1}{2\pi RC}$ | **Wien** | Siempre que aparezca OPAM no inversor + red RC Wien |
| $A = 3$ ($R_f = 2R_g$) | **Wien** | Condicion de ganancia, siempre fija |
| $f_0 = \dfrac{1}{2\pi RC\sqrt{6}}$ | **Desplaz. fase** | Amplificador inversor + 3 etapas RC |
| $A = 29$ ($R_f = 29R$) | **Desplaz. fase** | Condicion de ganancia, siempre fija |
| $f_0 = \dfrac{1}{2\pi\sqrt{LC_{eq}}}$ | **Colpitts** | Tanque LC con divisor capacitivo |
| $C_{eq} = \dfrac{C_1 C_2}{C_1 + C_2}$ | **Colpitts** | Capacidad equivalente serie |
| $f_0 = \dfrac{1}{2\pi\sqrt{L_{eq}C}}$ | **Hartley** | Tanque LC con divisor inductivo |
| $L_{eq} = L_1 + L_2$ | **Hartley** | Inductancia equivalente serie |
| $T = 0.693(R_A + 2R_B)C$ | **555** | Periodo de la onda cuadrada |
| $f = \dfrac{1.44}{(R_A + 2R_B)C}$ | **555** | Frecuencia directa |
| $D = \dfrac{R_A + R_B}{R_A + 2R_B}$ | **555** | Ciclo de trabajo |

### Trucos para el examen

- **Wien:** ganancia = 3, siempre. Si no, no oscila sinusoidalmente
- **Desplaz. fase:** factor $\sqrt{6}$ y ganancia = 29
- **Colpitts vs Hartley:** Colpitts = Capacidades, Hartley = Henrios (inductancias)
- **555:** el ciclo de trabajo basico siempre es $> 50\%$
- **Barkhausen:** si $|T| < 1$, muere; si $|T| > 1$, crece hasta saturar